# EEG Alzheimer — Full pipeline to create a RF classification for detection of alzheimer

1. Channel location (montage 10-20)
2. Re-referencing (average)
3. Filtering (1–100 Hz)
4. Bad channel + bad block removal
5. Epoching (fixed-length, resting-state)
6. Bin assignment (AD / CN label per epoch)
7. ICA decomposition
8. Removal of non-brain components (ocular, cardiac, muscle)
9. Epoch average per subject
10. Feature extraction — band power, latency proxies, amplitude, connectivity
11. Random Forest classifier (AD vs CN)

> `Runtime → Change runtime type → T4 GPU`  
> Total time to run: around 25 minute

## Install, and dowload of the dtaatset from OpenNeuro S3

In [18]:
%%capture
!pip install mne scikit-learn pandas numpy matplotlib scipy
!pip install awscli --upgrade
print('done')

In [19]:
import os
from pathlib import Path

DATA_ROOT = Path('/content/ds004504')

if DATA_ROOT.exists() and any(DATA_ROOT.rglob('*.set')):
    print('Data already present, no download.')
else:
    print('Downloading from OpenNeuro S3...')
    os.makedirs(DATA_ROOT, exist_ok=True)
    !aws s3 sync --no-sign-request s3://openneuro.org/ds004504 {DATA_ROOT} \
        --exclude "*" \
        --include "sub-*/eeg/*.set" \
        --include "sub-*/eeg/*.json" \
        --include "participants.tsv" \
        --no-progress
    n = len(list(DATA_ROOT.rglob('*.set')))
    print(f'Done — {n} .set files')

Data already present, no download.



## Imports and configuration

*   Filterin: Bandpass beetween 1 and 100, ica need high frequency for muscle detection (sbove 45)
*   Epoching: 4 sec, fixed
*   Peak-to-peak rejection threshold is 150 µV bc is cenrtanly not brain, EEG sits between 10–100 µV.

In [20]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch, hilbert
from scipy.stats import kurtosis as scipy_kurtosis, skew
from scipy.signal import find_peaks

import mne
from mne.preprocessing import ICA

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

DATA_ROOT   = Path('/content/ds004504')
RESULTS_DIR = Path('/content/results')
RESULTS_DIR.mkdir(exist_ok=True)

# Filtering and epoch
BANDPASS      = (1.0, 100.0)   # broad — ICA needs high freqs for muscle detection
SFREQ_TARGET  = 256

EPOCH_DUR     = 4.0
EPOCH_OVERLAP = 0.0
REJECT_PTP    = 150e-6

BANDS = {
    'delta': (0.5,  4),
    'theta': (4,    8),
    'alpha': (8,   13),
    'beta':  (13,  30),
    'gamma': (30,  45),
}

# Labels
LABEL_MAP = {'A': 1, 'C': 0}
LABEL_STR = {1: 'AD', 0: 'CN'}

print('Config ready')

Config ready


In [21]:
df = pd.read_csv(DATA_ROOT / 'participants.tsv', sep='\t')
print('All groups:', df['Group'].value_counts().to_dict())

df = df[df['Group'].isin(LABEL_MAP.keys())].copy()
df['label'] = df['Group'].map(LABEL_MAP)
print(f'Kept: AD={sum(df.label==1)}, CN={sum(df.label==0)}')
df.head()

All groups: {'A': 36, 'C': 29, 'F': 23}
Kept: AD=36, CN=29


,participant_id,Gender,Age,Group,MMSE,label
0,sub-001,F,57,A,16,1
1,sub-002,F,78,A,22,1
2,sub-003,M,70,A,14,1
3,sub-004,F,67,A,20,1
4,sub-005,M,70,A,22,1


## Step 1+2+3+4: Channel location, Re-reference, Filter 1–100 Hz, Bad channel/block removal
- Bad CHANNELS: flag channels with anomalous variance (MAD z-score > 5)
- Bad BLOCKS: annotate time segments with extreme amplitude

In [22]:
def find_eeg_file(data_root, subject_id):
    for pat in [
        f'derivatives/lossless/{subject_id}/eeg/*.set',
        f'{subject_id}/eeg/*.set',
        f'{subject_id}/eeg/*.edf',
        f'{subject_id}/eeg/*.bdf',
    ]:
        m = list(data_root.glob(pat))
        if m: return m[0]
    return None


def step1_channel_location(raw):

    if raw.get_montage() is None:
        try:
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage, on_missing='ignore')
        except Exception:
            pass
    return raw


def step2_rereference(raw):
    raw.set_eeg_reference('average', projection=True)
    raw.apply_proj()
    return raw


def step3_filter(raw):
    if raw.info['sfreq'] != SFREQ_TARGET:
        raw.resample(SFREQ_TARGET)

    raw.filter(*BANDPASS, fir_design='firwin', skip_by_annotation='edge')

    # Notch filter, 50hz powerline, EU standard)
    raw.notch_filter(freqs=50.0, fir_design='firwin')
    return raw


def step4_remove_bad_channels_and_blocks(raw):

    data = raw.get_data()

    #  Bad channels
    ch_var = data.var(axis=1)
    med    = np.median(ch_var)
    mad    = np.median(np.abs(ch_var - med)) + 1e-12
    bad_idx = np.where(np.abs(ch_var - med) / mad > 5)[0]
    if len(bad_idx):
        bads = [raw.ch_names[i] for i in bad_idx]
        raw.info['bads'].extend(list(set(bads) - set(raw.info['bads'])))
        raw.interpolate_bads(reset_bads=True)
        print(f'    [bad ch] interpolated: {bads}')

    #  Bad blocks
    sfreq    = int(raw.info['sfreq'])
    win_size = sfreq
    data_mean = data.mean(axis=0)
    n_wins   = len(data_mean) // win_size
    rms_wins = np.array([
        np.sqrt(np.mean(data_mean[i*win_size:(i+1)*win_size]**2))
        for i in range(n_wins)
    ])

    med_rms = np.median(rms_wins)
    mad_rms = np.median(np.abs(rms_wins - med_rms)) + 1e-12
    bad_wins = np.where(np.abs(rms_wins - med_rms) / mad_rms > 5)[0]

    if len(bad_wins):
        onsets     = bad_wins * win_size / sfreq
        durations  = np.ones(len(bad_wins))
        desc       = ['BAD_block'] * len(bad_wins)
        annot      = mne.Annotations(onset=onsets, duration=durations,
                                     description=desc,
                                     orig_time=raw.info['meas_date'])
        raw.set_annotations(raw.annotations + annot)
        print(f'    [bad blocks] annotated {len(bad_wins)} seconds')

    return raw


print(' Steps all done')

 Steps all done


## Epoching + Bin assignment
Take away all of the bad block, bad peak over 150µV, and divide by 4sec returing epoch.  FOr the bin, the people have eyes close so there is no stimulie, the only one put is the diagnostic group "Alzheimer vs Healthy"


In [23]:
def step5_epoch(raw):
    epochs = mne.make_fixed_length_epochs(
        raw,
        duration=EPOCH_DUR,
        overlap=EPOCH_OVERLAP,
        preload=True,
        reject_by_annotation=True,
    )
    epochs.drop_bad(reject=dict(eeg=REJECT_PTP))
    return epochs


def step6_bin_assignment(epochs, label, subject_id):
    n_ep = len(epochs)
    meta = pd.DataFrame({
        'subject':  [subject_id] * n_ep,
        'bin':      [LABEL_STR[label]] * n_ep,
        'label':    [label] * n_ep,
        'epoch_idx': np.arange(n_ep),
    })
    epochs.metadata = meta
    return epochs


print('Steps 5–6 done')

Steps 5–6 done


## Steps 7+8: ICA + remoove non-brain components
- OCULAR (eye blinks / movements):see in frontal electrodes are closest to the eyes and pick up blink potentials most strongly.

- CARDIAC (heartbeat / BCG):Component with very high kurtosis and "visible" every 1-sec lag > 0.3, bc heartbeating is regular.

- MUSCLE: high power ratio in 40–100 Hz vs 1–40 Hz (>3x)

In [24]:
def step7_ica(raw):
    n_components = min(15, len(raw.ch_names) - len(raw.info['bads']) - 1)
    if n_components < 2:
        return None, raw

    raw_hp = raw.copy().filter(1.0, None, fir_design='firwin')
    ica    = ICA(n_components=n_components, method='fastica',
                 random_state=42, max_iter=400)
    try:
        ica.fit(raw_hp)
    except Exception as e:
        print(f'    [ICA] fit failed: {e}')
        return None, raw

    return ica, raw_hp


def step8_remove_bad_components(raw, ica, raw_hp):
    if ica is None:
        return raw

    n_components = ica.n_components_
    mixing       = ica.mixing_matrix_
    sources      = ica.get_sources(raw_hp).get_data()
    sfreq        = raw.info['sfreq']
    exclude      = []

    n_mix_ch     = mixing.shape[0]
    ica_ch_names = [raw.ch_names[i] for i in range(n_mix_ch)]
    frontal = [ch for ch in ['Fp1', 'Fp2', 'FP1', 'FP2'] if ch in ica_ch_names]
    for ch_name in frontal:
        ch_idx   = ica_ch_names.index(ch_name)
        template = np.zeros(n_mix_ch)
        template[ch_idx] = 1.0
        for comp in range(n_components):
            corr = np.corrcoef(mixing[:, comp], template)[0, 1]
            if abs(corr) > 0.6 and comp not in exclude:
                exclude.append(comp)
                print(f'    [ICA] comp {comp:02d} → OCULAR  (r={corr:.2f} with {ch_name})')

    for comp in range(n_components):
        if comp in exclude: continue
        sig  = sources[comp]
        kurt = scipy_kurtosis(sig)
        if kurt > 10:
            lag = int(sfreq)
            if lag < len(sig):
                ac = np.corrcoef(sig[:-lag], sig[lag:])[0, 1]
                if ac > 0.3:
                    exclude.append(comp)
                    print(f'    [ICA] comp {comp:02d} → CARDIAC (kurt={kurt:.1f}, ac={ac:.2f})')

    for comp in range(n_components):
        if comp in exclude: continue
        sig = sources[comp]
        freqs_w, psd_w = welch(sig, fs=sfreq, nperseg=int(sfreq * 2))
        lo_idx  = np.where((freqs_w >= 1)  & (freqs_w <= 40))[0]
        hi_idx  = np.where((freqs_w >= 40) & (freqs_w <= 100))[0]
        if len(lo_idx) == 0 or len(hi_idx) == 0: continue
        ratio = psd_w[hi_idx].mean() / (psd_w[lo_idx].mean() + 1e-12)
        if ratio > 3.0:
            exclude.append(comp)
            print(f'    [ICA] comp {comp:02d} → MUSCLE  (HF/LF ratio={ratio:.1f})')

    if exclude:
        ica.exclude = list(set(exclude))
        ica.apply(raw)
        print(f'    [ICA] total removed: {len(ica.exclude)} components')
    else:
        print(f'    [ICA] no artifact components found')

    return raw


print('Steps 7–8 done')

Steps 7–8 done


## Epoch average + Feature extraction


| Group | Feature | Description |
|---|---|---|
| Band power | log abs power × 5 bands × n_ch | Welch PSD, absolute |
| Band power | relative power × 5 bands × n_ch | band / total |
| Amplitude | mean, std, peak-to-peak per channel | time-domain stats |
| Latency proxy | time-to-peak of alpha envelope per channel | ms analog |
| Spectral | peak frequency per channel | dominant freq in 1–45 Hz |
| Spectral | spectral edge frequency 95% per channel | SEF95 |
| Complexity | Hjorth mobility + complexity per channel | ERP classic stats |
| Connectivity | inter-channel alpha coherence (mean) | 1 scalar per subject |

In [25]:
def step9_average_epochs(epochs):
    data = epochs.get_data()
    avg  = data.mean(axis=0)
    return avg, data


def _hjorth(sig):
    diff1 = np.diff(sig)
    diff2 = np.diff(diff1)
    activity   = np.var(sig)
    mob_sig    = np.sqrt(np.var(diff1) / (activity + 1e-12))
    mob_diff   = np.sqrt(np.var(diff2) / (np.var(diff1) + 1e-12))
    complexity = mob_diff / (mob_sig + 1e-12)
    return mob_sig, complexity


def _spectral_edge_freq(freqs, psd, percent=0.95):
    cumsum = np.cumsum(psd)
    target = percent * cumsum[-1]
    idx    = np.searchsorted(cumsum, target)
    return freqs[min(idx, len(freqs)-1)]


def _alpha_envelope_latency(sig, sfreq):
    from scipy.signal import butter, filtfilt
    b, a    = butter(4, [8/(sfreq/2), 13/(sfreq/2)], btype='band')
    filt    = filtfilt(b, a, sig)
    envelope = np.abs(hilbert(filt))
    peak_idx = np.argmax(envelope)
    return peak_idx / sfreq * 1000


def step10_extract_features(avg, epochs_data, sfreq):
    n_ch    = avg.shape[0]
    n_bands = len(BANDS)
    nperseg = int(sfreq * 2)
    feats   = []


    abs_pow_all = np.zeros((len(epochs_data), n_ch, n_bands))
    rel_pow_all = np.zeros((len(epochs_data), n_ch, n_bands))

    for ep in range(len(epochs_data)):
        for ch in range(n_ch):
            freqs_w, psd = welch(epochs_data[ep, ch], fs=sfreq, nperseg=nperseg)
            total = psd.sum() + 1e-12
            for b, (_, (flo, fhi)) in enumerate(BANDS.items()):
                idx = np.where((freqs_w >= flo) & (freqs_w <= fhi))[0]
                abs_pow_all[ep, ch, b] = np.log1p(psd[idx].mean())
                rel_pow_all[ep, ch, b] = psd[idx].sum() / total


    feats.extend(abs_pow_all.mean(axis=0).flatten())
    feats.extend(rel_pow_all.mean(axis=0).flatten())


    amplitude_mean    = []
    amplitude_std     = []
    amplitude_ptp     = []
    latency_proxy     = []
    peak_freq         = []
    sef95             = []
    hjorth_mob        = []
    hjorth_complex    = []

    for ch in range(n_ch):
        sig = avg[ch]

        amplitude_mean.append(np.mean(np.abs(sig)))
        amplitude_std.append(np.std(sig))
        amplitude_ptp.append(np.ptp(sig))

        latency_proxy.append(_alpha_envelope_latency(sig, sfreq))

        freqs_w, psd = welch(sig, fs=sfreq, nperseg=nperseg)
        mask = (freqs_w >= 1) & (freqs_w <= 45)
        freqs_m, psd_m = freqs_w[mask], psd[mask]
        peak_freq.append(freqs_m[np.argmax(psd_m)])
        sef95.append(_spectral_edge_freq(freqs_m, psd_m, 0.95))

        mob, comp = _hjorth(sig)
        hjorth_mob.append(mob)
        hjorth_complex.append(comp)

    feats.extend(amplitude_mean)
    feats.extend(amplitude_std)
    feats.extend(amplitude_ptp)
    feats.extend(latency_proxy)
    feats.extend(peak_freq)
    feats.extend(sef95)
    feats.extend(hjorth_mob)
    feats.extend(hjorth_complex)


    alpha_lo, alpha_hi = BANDS['alpha']
    coh_values = []
    for ch1, ch2 in combinations(range(n_ch), 2):
        f, cxy = welch(avg[ch1] * avg[ch2], fs=sfreq, nperseg=nperseg)
        _, pxx = welch(avg[ch1], fs=sfreq, nperseg=nperseg)
        _, pyy = welch(avg[ch2], fs=sfreq, nperseg=nperseg)
        idx    = np.where((f >= alpha_lo) & (f <= alpha_hi))[0]
        with np.errstate(divide='ignore', invalid='ignore'):
            coh  = cxy[idx]**2 / (pxx[idx] * pyy[idx] + 1e-12)
        coh_values.append(np.mean(np.abs(coh)))
    feats.append(np.mean(coh_values))

    return np.array(feats)


print('Steps 9–10 done')

Steps 9–10 done


## Epoch average per subject

In [26]:
X_list, y_list, subject_ids = [], [], []
epoch_counts  = []
failed_subjects = []

for i, (_, row) in enumerate(df.iterrows()):
    sub_id = row['participant_id']
    label  = row['label']

    eeg_path = find_eeg_file(DATA_ROOT, sub_id)
    if eeg_path is None:
        print(f'[{i+1:02d}] {sub_id}: no file — skip')
        continue

    print(f'\n[{i+1:02d}] {sub_id} ({LABEL_STR[label]})')

    try:
        raw = mne.io.read_raw(str(eeg_path), preload=True)

        raw = step1_channel_location(raw)
        raw = step2_rereference(raw)
        raw = step3_filter(raw)
        raw = step4_remove_bad_channels_and_blocks(raw)
        sfreq = raw.info['sfreq']

        ica, raw_hp = step7_ica(raw)
        raw = step8_remove_bad_components(raw, ica, raw_hp)

        epochs = step5_epoch(raw)
        if len(epochs) == 0:
            print('  no valid epochs — skip')
            failed_subjects.append((sub_id, 'no valid epochs'))
            continue
        epochs = step6_bin_assignment(epochs, label, sub_id)
        print(f'  epochs: {len(epochs)} valid')

        avg, epochs_data = step9_average_epochs(epochs)
        feats = step10_extract_features(avg, epochs_data, sfreq)

        X_list.append(feats)
        y_list.append(label)
        subject_ids.append(sub_id)
        epoch_counts.append(len(epochs))
        print(f'  features: {feats.shape[0]} dims  ✓')

    except Exception as e:
        import traceback
        print(f'  ✗ ERROR: {e}')
        print('  ' + traceback.format_exc().splitlines()[-2])
        failed_subjects.append((sub_id, str(e)))
        continue

X = np.array(X_list)
y = np.array(y_list)

print(f'\n═══ Dataset ready: {X.shape[0]} subjects × {X.shape[1]} features')
print(f'    AD={sum(y==1)}, CN={sum(y==0)}')
print(f'    Avg epochs/subject: {np.mean(epoch_counts):.0f}')
if failed_subjects:
    print(f'\n    Failed ({len(failed_subjects)}):')
    for s, reason in failed_subjects:
        print(f'      {s}: {reason}')


[01] sub-001 (AD)
    [bad ch] interpolated: ['Fp1', 'Fp2', 'F7']
    [ICA] comp 00 → OCULAR  (r=0.94 with Fp1)
    [ICA] comp 02 → OCULAR  (r=0.91 with Fp1)
    [ICA] comp 03 → OCULAR  (r=-0.67 with Fp1)
    [ICA] comp 04 → OCULAR  (r=0.67 with Fp1)
    [ICA] comp 05 → OCULAR  (r=0.83 with Fp1)
    [ICA] comp 08 → OCULAR  (r=-0.67 with Fp1)
    [ICA] comp 01 → OCULAR  (r=0.83 with Fp2)
    [ICA] comp 06 → OCULAR  (r=0.62 with Fp2)
    [ICA] total removed: 8 components
  epochs: 146 valid
  ✗ ERROR: name 'combinations' is not defined
                      ^^^^^^^^^^^^

[02] sub-002 (AD)
    [bad ch] interpolated: ['Fp1', 'Fp2']
    [ICA] comp 00 → OCULAR  (r=-0.99 with Fp1)
    [ICA] comp 04 → OCULAR  (r=-0.79 with Fp1)
    [ICA] comp 11 → OCULAR  (r=0.64 with Fp1)
    [ICA] comp 01 → OCULAR  (r=-0.68 with Fp2)
    [ICA] comp 02 → OCULAR  (r=0.67 with Fp2)
    [ICA] comp 03 → OCULAR  (r=0.63 with Fp2)
    [ICA] comp 06 → OCULAR  (r=-0.61 with Fp2)
    [ICA] total removed: 7 components

IndexError: tuple index out of range

## Save

In [27]:
feat_df = pd.DataFrame(X)
feat_df.insert(0, 'subject', subject_ids)
feat_df.insert(1, 'label',   y)
feat_df.insert(2, 'group',   [LABEL_STR[l] for l in y])
feat_df.insert(3, 'n_epochs', epoch_counts)
feat_df.to_csv(RESULTS_DIR / 'features.csv', index=False)
print(f' Saved {feat_df.shape} → {RESULTS_DIR}/features.csv')

✓ Saved (0, 5) → /content/results/features.csv


## Alzheimer or not, comparison of caratteristic

In [29]:
n_bands    = len(BANDS)
band_names = list(BANDS.keys())
n_ch = (X.shape[1] - 1) // (2 * n_bands + 8)

abs_feats   = X[:, :n_bands * n_ch].reshape(len(X), n_ch, n_bands)
mean_by_band = abs_feats.mean(axis=1)

ad_mean = mean_by_band[y==1].mean(axis=0)
cn_mean = mean_by_band[y==0].mean(axis=0)
ad_sem  = mean_by_band[y==1].std(axis=0) / np.sqrt((y==1).sum())
cn_sem  = mean_by_band[y==0].std(axis=0) / np.sqrt((y==0).sum())

x, w = np.arange(n_bands), 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x-w/2, cn_mean, w, yerr=cn_sem, label='Healthy (CN)', color='#4C72B0', capsize=4)
ax.bar(x+w/2, ad_mean, w, yerr=ad_sem, label='Alzheimer (AD)', color='#C44E52', capsize=4)
ax.set_xticks(x); ax.set_xticklabels(band_names)
ax.set_ylabel('Mean log band power'); ax.set_title('Band Power: AD vs CN')
ax.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'band_power.png', dpi=150); plt.show()

IndexError: tuple index out of range

In [ ]:
offset     = 2 * n_ch * n_bands
amp_ptp    = X[:, offset + 2*n_ch : offset + 3*n_ch].mean(axis=1)
latency    = X[:, offset + 3*n_ch : offset + 4*n_ch].mean(axis=1)
peak_freq  = X[:, offset + 4*n_ch : offset + 5*n_ch].mean(axis=1)
sef        = X[:, offset + 5*n_ch : offset + 6*n_ch].mean(axis=1)
coherence  = X[:, -1]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, vals, title in zip(
    axes,
    [amp_ptp, latency, peak_freq, sef],
    ['Peak-to-peak amplitude (µV)', 'Alpha latency proxy (ms)',
     'Peak frequency (Hz)', 'SEF95 (Hz)']
):
    for label_val, color, name in [(0, '#4C72B0', 'CN'), (1, '#C44E52', 'AD')]:
        v = vals[y == label_val]
        ax.bar(name, v.mean(), yerr=v.std()/np.sqrt(len(v)),
               color=color, capsize=5, alpha=0.85)
    ax.set_title(title, fontsize=9)

plt.suptitle('ERP-style Features: AD vs CN', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'erp_features.png', dpi=150)
plt.show()

## RF

In [ ]:
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']

cv_results = cross_validate(clf, X, y, cv=cv, scoring=scoring)

print('── 5-Fold CV Results ─────────────────────────')
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f'  {metric:12s}: {scores.mean():.3f} ± {scores.std():.3f}')
print('──────────────────────────────────────────────')

clf.fit(X, y)
print(' Done, full dataset used')

## Feature importance

In [ ]:
metrics = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']
means   = [cv_results[f'test_{m}'].mean() for m in metrics]
stds    = [cv_results[f'test_{m}'].std()  for m in metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(metrics, means, yerr=stds, capsize=5, color='#55A868', alpha=0.85)
axes[0].axhline(0.5, color='red', linestyle='--', lw=0.8, label='chance')
axes[0].set_ylim(0, 1.05); axes[0].set_title('5-Fold CV Performance'); axes[0].legend()

rf = clf.named_steps['rf']
importances = rf.feature_importances_
ch_labels   = [f'ch{i}' for i in range(n_ch)]
feat_names  = (
    [f'abs_{b}_{c}' for c in ch_labels for b in band_names] +
    [f'rel_{b}_{c}' for c in ch_labels for b in band_names] +
    [f'amp_mean_{c}' for c in ch_labels] +
    [f'amp_std_{c}'  for c in ch_labels] +
    [f'amp_ptp_{c}'  for c in ch_labels] +
    [f'latency_{c}'  for c in ch_labels] +
    [f'peak_f_{c}'   for c in ch_labels] +
    [f'sef95_{c}'    for c in ch_labels] +
    [f'hjorth_m_{c}' for c in ch_labels] +
    [f'hjorth_c_{c}' for c in ch_labels] +
    ['alpha_coherence']
)
if len(feat_names) != len(importances):
    feat_names = [f'feat_{i}' for i in range(len(importances))]

top_k = 25
idx   = np.argsort(importances)[::-1][:top_k]
axes[1].barh(range(top_k), importances[idx][::-1], color='#4C72B0')
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([feat_names[i] for i in idx[::-1]], fontsize=7)
axes[1].set_title('Top-25 Feature Importances')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'results_summary.png', dpi=150)
plt.show()

## Save model + download all results

In [ ]:
import zipfile
from google.colab import files

joblib.dump(clf, RESULTS_DIR / 'rf_model.pkl')
print('✓ Model saved')

zip_path = '/content/results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in RESULTS_DIR.iterdir():
        zf.write(f, f.name)

files.download(zip_path)
print('✓ results.zip downloaded')

## Bonus: Ablation study, wich feature realy matter?

In [ ]:
from sklearn.model_selection import cross_val_score
import shap

n_bp = n_ch * n_bands

feature_groups = {
    'rel_bandpower_only':     X[:, n_bp : 2*n_bp],
    'abs_bandpower_only':     X[:, :n_bp],
    'bandpower_both':         X[:, :2*n_bp],
    'erp_features_only':      X[:, 2*n_bp:-1],
    'no_hjorth':              np.hstack([X[:, :2*n_bp+6*n_ch], X[:,-1:]]),
    'all_features':           X,
}

results_ablation = {}
for name, Xsub in feature_groups.items():
    scores = cross_val_score(
        Pipeline([('sc', StandardScaler()),
                  ('rf', RandomForestClassifier(500, class_weight='balanced',
                                                random_state=42, n_jobs=-1))]),
        Xsub, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='roc_auc'
    )
    results_ablation[name] = (scores.mean(), scores.std())
    print(f'{name:30s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}')

explainer   = shap.TreeExplainer(clf.named_steps['rf'])
X_scaled    = clf.named_steps['scaler'].transform(X)
shap_values = explainer.shap_values(X_scaled)


if isinstance(shap_values, list):
    sv_ad = shap_values[1]
else:
    sv_ad = shap_values[:, :, 1]

feat_names_trimmed = feat_names[:X_scaled.shape[1]]

shap.summary_plot(
    sv_ad,
    X_scaled,
    feature_names=feat_names_trimmed,
    max_display=20,
    show=False
)
plt.title('SHAP — AD class')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'shap_summary.png', dpi=150)
plt.show()